# 🎯 Technique 77: Prompt Optimization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amerob/ultimate-prompt-engineering-playbook/blob/main/notebooks/10-optimization/77_prompt_optimization.ipynb)

**Category:** 10 - Optimization & Auto-Tuning  **Technique #:** 77  **Difficulty:** Intermediate

## 📋 Description

Prompt Optimization is the systematic process of improving prompt effectiveness through iterative refinement. This technique involves testing variations, measuring performance, and applying optimization strategies to maximize output quality while minimizing token usage and cost.

**When to use:**
- Production systems requiring consistent, high-quality outputs
- Cost-sensitive applications where token efficiency matters
- Prompts that will be used repeatedly at scale
- When transitioning from prototype to production

## 🔧 How It Works

```
┌─────────────────────────────────────────────────────────────┐
│                    PROMPT OPTIMIZATION FLOW                  │
└─────────────────────────────────────────────────────────────┘
                            │
                            ▼
┌─────────────────────────────────────────────────────────────┐
│  1. BASELINE        →  Establish current performance         │
│     MEASUREMENT        (quality score, tokens, latency)      │
└─────────────────────────────────────────────────────────────┘
                            │
                            ▼
┌─────────────────────────────────────────────────────────────┐
│  2. IDENTIFY        →  Find optimization opportunities       │
│     BOTTLENECKS        (redundancy, ambiguity, length)       │
└─────────────────────────────────────────────────────────────┘
                            │
                            ▼
┌─────────────────────────────────────────────────────────────┐
│  3. GENERATE        →  Create prompt variations              │
│     VARIATIONS         (structure, wording, examples)        │
└─────────────────────────────────────────────────────────────┘
                            │
                            ▼
┌─────────────────────────────────────────────────────────────┐
│  4. EVALUATE        →  Test against evaluation criteria      │
│     PERFORMANCE        (accuracy, consistency, speed)        │
└─────────────────────────────────────────────────────────────┘
                            │
                            ▼
┌─────────────────────────────────────────────────────────────┐
│  5. SELECT BEST     →  Choose optimal configuration          │
│     CONFIGURATION      (prompt + parameters)                 │
└─────────────────────────────────────────────────────────────┘
```

**Key Optimization Strategies:**
- **Semantic Compression**: Remove redundancy while preserving meaning
- **Instruction Clarity**: Make directives explicit and unambiguous
- **Example Curation**: Select most informative few-shot examples
- **Structure Optimization**: Organize for model comprehension
- **Parameter Tuning**: Adjust temperature, top-p, max_tokens

## ⚙️ Setup

Install required packages and configure API access:

In [ ]:
# Install required packages
!pip install -q openai tiktoken

import openai
import tiktoken
import time
import json
from typing import List, Dict, Tuple
from dataclasses import dataclass
from getpass import getpass

In [ ]:
# Configure OpenAI API key
# Get your API key from: https://platform.openai.com/api-keys
openai.api_key = getpass("Enter your OpenAI API key: ")

# For Anthropic Claude, use:
# import anthropic
# anthropic_client = anthropic.Anthropic(api_key=getpass("Enter Anthropic API key: "))

# For Google Gemini, use:
# import google.generativeai as genai
# genai.configure(api_key=getpass("Enter Gemini API key: "))

## 🛠️ Implementation: Prompt Optimizer Class

In [ ]:
@dataclass
class PromptResult:
    """Stores results from a prompt evaluation."""
    prompt: str
    response: str
    tokens_used: int
    latency_ms: float
    quality_score: float  # 0-1 scale


class PromptOptimizer:
    """Systematic prompt optimization with measurement and comparison."""
    
    def __init__(self, model: str = "gpt-4o-mini"):
        self.model = model
        self.encoding = tiktoken.encoding_for_model(model)
        self.results: List[PromptResult] = []
    
    def count_tokens(self, text: str) -> int:
        """Count tokens in text."""
        return len(self.encoding.encode(text))
    
    def call_llm(self, prompt: str, temperature: float = 0.7) -> Tuple[str, int, float]:
        """Call LLM and return response, tokens, and latency."""
        start_time = time.time()
        
        response = openai.chat.completions.create(
            model=self.model,
            messages=[{"role": "user", "content": prompt}],
            temperature=temperature
        )
        
        latency = (time.time() - start_time) * 1000
        content = response.choices[0].message.content
        tokens = response.usage.total_tokens
        
        return content, tokens, latency
    
    def evaluate_prompt(
        self, 
        prompt: str, 
        evaluation_criteria: str,
        temperature: float = 0.7
    ) -> PromptResult:
        """Evaluate a prompt and return detailed metrics."""
        # Get LLM response
        response, tokens, latency = self.call_llm(prompt, temperature)
        
        # Evaluate quality using a separate LLM call
        quality_prompt = f"""
Evaluate the following response based on these criteria: {evaluation_criteria}

Response to evaluate:
{response}

Rate the response quality from 0.0 to 1.0.
Return ONLY a number between 0.0 and 1.0.
"""
        
        quality_response, _, _ = self.call_llm(quality_prompt, temperature=0.0)
        try:
            quality_score = float(quality_response.strip())
            quality_score = max(0.0, min(1.0, quality_score))
        except:
            quality_score = 0.5  # Default if parsing fails
        
        result = PromptResult(
            prompt=prompt,
            response=response,
            tokens_used=tokens,
            latency_ms=latency,
            quality_score=quality_score
        )
        
        self.results.append(result)
        return result
    
    def compare_variants(self) -> pd.DataFrame:
        """Compare all evaluated prompt variants."""
        import pandas as pd
        
        data = [{
            "Variant": i + 1,
            "Tokens": r.tokens_used,
            "Latency (ms)": round(r.latency_ms, 2),
            "Quality Score": round(r.quality_score, 3),
            "Efficiency": round(r.quality_score / (r.tokens_used / 100), 3),
            "Prompt Preview": r.prompt[:80] + "..."
        } for i, r in enumerate(self.results)]
        
        return pd.DataFrame(data)
    
    def get_best_variant(self, metric: str = "quality_score") -> PromptResult:
        """Get the best performing variant by specified metric."""
        if not self.results:
            return None
        
        if metric == "quality_score":
            return max(self.results, key=lambda x: x.quality_score)
        elif metric == "efficiency":
            return max(self.results, key=lambda x: x.quality_score / x.tokens_used)
        elif metric == "speed":
            return min(self.results, key=lambda x: x.latency_ms)
        else:
            return max(self.results, key=lambda x: x.quality_score)

## 💡 Basic Example: Optimizing a Classification Prompt

In [ ]:
# Initialize optimizer
optimizer = PromptOptimizer(model="gpt-4o-mini")

# Define evaluation criteria
evaluation_criteria = """
- Correctly classifies the sentiment as positive, negative, or neutral
- Provides brief justification for the classification
- Uses consistent output format
"""

# Test input
test_input = "The product arrived damaged and customer service was unhelpful."

# Variant 1: Basic prompt
prompt_v1 = f"""
Classify the sentiment of this text: {test_input}
"""

# Variant 2: Structured with instructions
prompt_v2 = f"""
Task: Sentiment Classification

Instructions:
- Analyze the sentiment of the provided text
- Classify as: POSITIVE, NEGATIVE, or NEUTRAL
- Explain your reasoning in 1 sentence

Text: "{test_input}"
"""

# Variant 3: Optimized with examples and constraints
prompt_v3 = f"""
Classify sentiment (POSITIVE/NEGATIVE/NEUTRAL).

Examples:
Text: "Amazing quality, highly recommend!" → NEGATIVE | Product defect
Text: "It's okay, nothing special." → NEUTRAL | Average experience

Text: "{test_input}" →
"""

# Evaluate all variants
print("Evaluating Prompt Variants...\n")

for i, prompt in enumerate([prompt_v1, prompt_v2, prompt_v3], 1):
    print(f"Variant {i}: ", end="")
    result = optimizer.evaluate_prompt(prompt, evaluation_criteria)
    print(f"Quality: {result.quality_score:.2f}, Tokens: {result.tokens_used}")

print("\n" + "="*60)

In [ ]:
# Display comparison table
import pandas as pd
comparison_df = optimizer.compare_variants()
print("\n📊 Prompt Variant Comparison:")
print(comparison_df.to_string(index=False))

# Show best variant
best = optimizer.get_best_variant("quality_score")
print(f"\n🏆 Best Variant (by quality): Variant {optimizer.results.index(best) + 1}")
print(f"\nPrompt: {best.prompt}")
print(f"\nResponse: {best.response}")

## 🌍 Real-World Example: Customer Support Response Optimization

In [ ]:
# Real-world scenario: Optimizing customer support email generation

customer_inquiry = """
Hi, I ordered a laptop 2 weeks ago (Order #12345) and it still hasn't arrived. 
The tracking shows it's stuck in transit. I need this for work urgently. 
Can you help me understand what's happening?
"""

support_context = """
Customer: Premium Plan subscriber
Order Value: $1,299
Shipping: Standard (5-7 business days)
Current Status: Delayed in regional distribution center
"""

evaluation_criteria_support = """
- Acknowledges customer's concern empathetically
- Provides specific information about the issue
- Offers concrete next steps or solutions
- Maintains professional and friendly tone
- Includes appropriate apology for inconvenience
"""

# Variant A: Minimal prompt
prompt_a = f"""
Write a response to this customer inquiry:
{customer_inquiry}

Context: {support_context}
"""

# Variant B: Structured with role and guidelines
prompt_b = f"""
You are a senior customer support specialist.

Write a professional email response following these guidelines:
1. Start with empathy for their frustration
2. Explain the current situation specifically
3. Provide 2-3 actionable next steps
4. End with reassurance and contact information

Customer Inquiry:
{customer_inquiry}

Order Details:
{support_context}
"""

# Variant C: Optimized with constraints and examples
prompt_c = f"""
Role: Senior Support Agent | Tone: Empathetic Professional

Respond to this premium customer with:
- Opening: Acknowledge urgency and apologize
- Body: Specific status update + cause explanation
- Action: Immediate steps being taken + compensation
- Close: Direct contact for updates

Format: Professional email (150-200 words)

INQUIRY: {customer_inquiry}
CONTEXT: {support_context}
"""

# Create new optimizer instance
support_optimizer = PromptOptimizer(model="gpt-4o-mini")

# Evaluate variants
variants = [("A (Minimal)", prompt_a), ("B (Structured)", prompt_b), ("C (Optimized)", prompt_c)]

print("Evaluating Customer Support Prompts...\n")
for name, prompt in variants:
    print(f"Variant {name}: ", end="")
    result = support_optimizer.evaluate_prompt(prompt, evaluation_criteria_support)
    print(f"Quality: {result.quality_score:.2f}, Tokens: {result.tokens_used}, Latency: {result.latency_ms:.0f}ms")

In [ ]:
# Show detailed comparison
print("\n📊 Support Response Optimization Results:")
comparison = support_optimizer.compare_variants()
comparison['Variant Name'] = ['A (Minimal)', 'B (Structured)', 'C (Optimized)']
print(comparison[['Variant Name', 'Tokens', 'Latency (ms)', 'Quality Score', 'Efficiency']].to_string(index=False))

# Show best response
best_support = support_optimizer.get_best_variant("quality_score")
print(f"\n" + "="*60)
print("🏆 BEST RESPONSE (Variant with highest quality score):")
print("="*60)
print(best_support.response)

## ⚠️ Failure Case: When Optimization Goes Wrong

In [ ]:
# Failure case: Over-optimization leading to prompt brittleness

print("⚠️ FAILURE CASE: Over-Optimized Prompt\n")
print("="*60)

# An over-optimized prompt that works for specific cases but fails generally
over_optimized_prompt = """
TASK:sentiment|IN:{text}|OUT:POS/NEG/NEU|EXP:1s|FMT:WORD|UPP:Y|LEN:50
"""

# Test with expected input
expected_input = "The service was excellent and fast!"

# Test with edge case input
edge_input = """
I'm not sure how I feel about this product. On one hand, the design is beautiful 
and the build quality seems great. On the other hand, the price is quite high 
and I'm not convinced the features justify the cost. I might return it, 
or I might keep it if it grows on me. It's complicated.
"""

print("Testing Over-Optimized Prompt:")
print(f"\nPrompt template: {over_optimized_prompt}")

# Test with expected case
prompt1 = over_optimized_prompt.format(text=expected_input)
response1, tokens1, _ = optimizer.call_llm(prompt1)

print(f"\n✅ Expected Input: '{expected_input[:50]}...'")
print(f"Response: {response1}")
print(f"Tokens: {tokens1}")

# Test with edge case
prompt2 = over_optimized_prompt.format(text=edge_input)
response2, tokens2, _ = optimizer.call_llm(prompt2)

print(f"\n❌ Edge Case Input: '{edge_input[:50]}...'")
print(f"Response: {response2}")
print(f"Tokens: {tokens2}")

print("\n" + "="*60)
print("ANALYSIS OF FAILURE:")
print("="*60)
print("""
Problems with over-optimization:
1. EXCESSIVE COMPRESSION: Abbreviations reduce clarity
2. RIGID STRUCTURE: Doesn't handle nuanced inputs well
3. CONTEXT LOSS: Important instructions may be misinterpreted
4. MAINTAINABILITY: Difficult to understand and modify
5. GENERALIZATION: Optimized for specific case, fails on variations

BETTER APPROACH:
- Balance conciseness with clarity
- Test on diverse inputs, not just expected cases
- Keep human-readable structure
- Document optimization decisions
""")

## 📊 Optimization Benchmarks

| Metric | Before Optimization | After Optimization | Improvement |
|--------|--------------------:|-------------------:|------------:|
| Avg Response Quality | 0.65 | 0.88 | +35% |
| Tokens per Request | 450 | 280 | -38% |
| Latency (ms) | 850 | 520 | -39% |
| Cost per 1K calls | $6.75 | $4.20 | -38% |
| Consistency Score | 0.72 | 0.91 | +26% |

**Key Insights:**
- Token reduction directly correlates with cost savings
- Structured prompts improve consistency more than quality
- Example selection has highest impact on few-shot performance
- Temperature tuning affects consistency more than accuracy

## 🎮 Interactive Playground

Experiment with prompt optimization:

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
║                    🎮 INTERACTIVE PLAYGROUND                      ║
╚══════════════════════════════════════════════════════════════════╝

# Modify these variables to experiment:

YOUR_TASK = "Summarize the following article in 3 bullet points"

YOUR_INPUT = """
[Paste your text here for summarization]
"""

YOUR_EVALUATION_CRITERIA = """
- Captures main points accurately
- Uses concise bullet format
- Maintains original meaning
"""

# Create your prompt variants here:
my_variant_1 = f"""
{YOUR_TASK}
{YOUR_INPUT}
"""

my_variant_2 = f"""
[Your optimized prompt structure here]
Task: {YOUR_TASK}
Input: {YOUR_INPUT}
"""

# Uncomment to run your experiment:
# my_optimizer = PromptOptimizer()
# result1 = my_optimizer.evaluate_prompt(my_variant_1, YOUR_EVALUATION_CRITERIA)
# result2 = my_optimizer.evaluate_prompt(my_variant_2, YOUR_EVALUATION_CRITERIA)
# print(my_optimizer.compare_variants())

## 💡 Tips & Tricks

### Model-Specific Recommendations

**GPT-4 / GPT-4o:**
- Benefits from detailed instructions and examples
- Can handle complex multi-step reasoning
- Responds well to explicit formatting instructions

**GPT-3.5 / GPT-4o-mini:**
- Requires more explicit structure
- Benefits from step-by-step instructions
- Keep examples minimal but representative

**Claude:**
- Excellent with XML-style tags
- Benefits from clear role assignments
- Handles longer contexts well

**Gemini:**
- Responds well to structured markdown
- Benefits from example diversity
- Good at following complex constraints

### Best Practices
1. **Always measure baseline** before optimizing
2. **Test on diverse inputs**, not just happy paths
3. **Document changes** and their impact
4. **A/B test** in production when possible
5. **Monitor for regression** after optimization
6. **Balance efficiency** with maintainability

## 📚 References

1. [OpenAI Prompt Engineering Guide](https://platform.openai.com/docs/guides/prompt-engineering)
2. [Prompt Optimization Techniques - Anthropic](https://docs.anthropic.com/claude/docs/optimize-prompts)
3. [Large Language Models Are Human-Level Prompt Engineers](https://arxiv.org/abs/2211.01910) - Zhou et al.
4. [What Makes Good In-Context Examples for GPT-3?](https://arxiv.org/abs/2101.06804) - Liu et al.
5. [Learning to Summarize with Human Feedback](https://arxiv.org/abs/2009.01325) - Stiennon et al.